*Goal of this notebook:**
1. Load MIMIC-IV demo CSVs into a local **SQLite** database (our Relational DB layer)
2. Build a **FAISS** vector index from medical guideline text (our Vector DB layer)
3. Write and test raw queries on both — so we understand exactly what the agents will be working with

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import faiss
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [ ]:
import os
# Run this to find the exact folder name
for item in os.listdir("."):
    print(item)

In [ ]:
# IN THIS FOLDER WE ARE BASICALLY CONFIGURING THE PATHS TO THE MIMIC CSV FILES, THE SQLITE DB WE WILL CREATE, AND THE FAISS INDEX FILE. MAKE SURE TO ADJUST THE MIMIC_DIR PATH IF YOUR FOLDER NAME IS DIFFERENT.
MIMIC_DIR = Path("./mimic-iv-clinical-database-demo-2.2")
DB_PATH     = Path("./mira_data/mimic.db")       # SQLite DB we will create
FAISS_PATH  = Path("./mira_data/medical_faiss.index")  # FAISS index file
META_PATH   = Path("./mira_data/faiss_metadata.pkl")   # chunk text + source

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
print("✅ Paths configured")
print(f"   MIMIC CSVs : {MIMIC_DIR.resolve()}")
print(f"   SQLite DB  : {DB_PATH.resolve()}")
print(f"   FAISS Index: {FAISS_PATH.resolve()}")

In [ ]:
# List all CSVs in the MIMIC directory tree
# IN THIS CELL WE ARE JUST CHECKING WHAT IS INSIDE MIMIC FOLDER ( HOW MANY CSV FILES )
all_csvs = list(MIMIC_DIR.rglob("*.csv")) + list(MIMIC_DIR.rglob("*.csv.gz"))

print(f"Found {len(all_csvs)} CSV files:\n")
for f in sorted(all_csvs):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.relative_to(MIMIC_DIR)}  ({size_mb:.1f} MB)")

In [ ]:
# Map: table_name → relative path inside MIMIC_DIR
# Adjust subfolder names (hosp/icu) based on what your ls showed above
# We focus on the **5 most clinically useful tables** for MIRA

# IN THIS CELL WE WILL LOAD THE NECESSARY TABLES INTO SQLITE
# And in this cell we also connected the sql database to the sqlite3 library and then we are loading the necessary tables into the sqlite database.

TABLE_MAP = { # this will basically load the necessary tables into the sqlite database
    "patients"      : "hosp/patients.csv",
    "admissions"    : "hosp/admissions.csv",
    "diagnoses_icd" : "hosp/diagnoses_icd.csv",
    "labevents"     : "hosp/labevents.csv",
    "d_labitems"    : "hosp/d_labitems.csv",
}

conn = sqlite3.connect(DB_PATH) # connecting the sqlite database

for table_name, rel_path in TABLE_MAP.items():
    csv_path = MIMIC_DIR / rel_path
    
    # Try .csv.gz if plain .csv not found
    if not csv_path.exists():
        csv_path = MIMIC_DIR / (rel_path + ".gz")
    
    if not csv_path.exists():
        print(f"⚠️  SKIPPED {table_name} — file not found at {csv_path}")
        continue
    
    print(f"📥 Loading {table_name}...", end=" ")
    df = pd.read_csv(csv_path, low_memory=False)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"✅ {len(df):,} rows → SQLite")

conn.commit()
print("\n🎉 All tables loaded into SQLite!")

In [ ]:
 # IN THIS CELL WE ARE CHECKING THE TABLES AND THEIR SCHEMAS IN THE SQLITE DB TO MAKE SURE EVERYTHING IS LOADED CORRECTLY
cursor = conn.cursor()

# List all tables
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print(f"Tables in DB: {[t[0] for t in tables]}\n")

# Show schema for each
for (table,) in tables:
    cols = cursor.execute(f"PRAGMA table_info({table})").fetchall()
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    col_names = [c[1] for c in cols]
    print(f"📋 {table} ({count:,} rows)")
    print(f"   Columns: {col_names}")
    print()

In [ ]:
def run_query(sql: str, label: str = "") -> pd.DataFrame:
    """Helper: run SQL and display results. This is what Agent 1 will call."""
    print(f"{'='*60}")
    if label:
        print(f"🔍 {label}")
    print(f"SQL: {sql.strip()}")
    print(f"{'='*60}")
    df = pd.read_sql_query(sql, conn)
    print(df.to_string(index=False))
    print(f"\n→ {len(df)} row(s) returned\n")
    return df

# ── Query 1: Basic patient demographics ──────────────────────────────────────
run_query("""
    SELECT subject_id, gender, anchor_age
    FROM patients
    LIMIT 10
""", "Patient demographics sample")

# ── Query 2: Recent admissions with diagnosis (FIXED) ────────────────────────
run_query("""
    SELECT a.subject_id, a.hadm_id, a.admittime, 
           a.admission_type, a.discharge_location, d.icd_code AS primary_diagnosis_code
    FROM admissions a
    LEFT JOIN diagnoses_icd d ON a.hadm_id = d.hadm_id AND d.seq_num = 1
    ORDER BY a.admittime DESC
    LIMIT 10
""", "Recent admissions")

# ── Query 3: Lab results for a specific patient ────────────────────────────────
run_query("""
    SELECT l.subject_id, d.label AS lab_name,
           l.value, l.valueuom AS unit, l.flag, l.charttime
    FROM labevents l
    JOIN d_labitems d ON l.itemid = d.itemid
    WHERE l.subject_id = (SELECT subject_id FROM patients LIMIT 1)
    ORDER BY l.charttime DESC
    LIMIT 15
""", "Lab results for first patient")

# ── Query 4: Patients with ABNORMAL lab flags (clinical triage relevant) ──────
run_query("""
    SELECT p.subject_id, p.gender, p.anchor_age,
           d.label AS lab_name, l.value, l.valueuom, l.flag
    FROM labevents l
    JOIN patients p ON l.subject_id = p.subject_id
    JOIN d_labitems d ON l.itemid = d.itemid
    WHERE l.flag = 'abnormal'
    ORDER BY l.charttime DESC
    LIMIT 20
""", "Patients with ABNORMAL lab results (triage priority)")

In [ ]:
# In this cell we have basically created an tool which wil covert the sql schema into text so that it can be used by the agent later onwards.
# This will help the agent to understannd the database and also help the agent to generate the sql queries based on the user prompt and the database schema.
def get_schema_string(conn: sqlite3.Connection) -> str:
    """Generates a human-readable schema string for LLM system prompts."""
    cursor = conn.cursor()
    tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    
    schema_parts = ["DATABASE SCHEMA (SQLite):\n"]
    
    for (table,) in tables:
        cols = cursor.execute(f"PRAGMA table_info({table})").fetchall()
        sample = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 2", conn)
        
        schema_parts.append(f"TABLE: {table}")
        for col in cols:
            schema_parts.append(f"  - {col[1]} ({col[2]})")
        schema_parts.append(f"  Sample row: {sample.iloc[0].to_dict() if len(sample) > 0 else 'empty'}")
        schema_parts.append("")
    
    return "\n".join(schema_parts)

DB_SCHEMA = get_schema_string(conn)

# Save to file so other notebooks can import it
schema_path = Path("./mira_data/db_schema.txt")
schema_path.write_text(DB_SCHEMA)

print(DB_SCHEMA)

In [ ]:
# In this cell we have basically created a corpus of medical guidelines that will be used by the agent later on to provide evidence-based recommendations. 
# Each guideline includes the source, topic, and key text. This will help the agent to generate more accurate and clinically relevant responses based on established medical knowledge.
# so this cell is basically just for additioinal medical knowledge to agent so that it can give more accurate responses.
MEDICAL_GUIDELINES = [
    # ── SEPSIS ──────────────────────────────────────────────────────────────
    {
        "source": "Surviving Sepsis Campaign 2021",
        "topic": "Sepsis Definition",
        "text": "Sepsis is defined as life-threatening organ dysfunction caused by a dysregulated host response to infection. Organ dysfunction is identified by an acute change in total SOFA score ≥2 points. Septic shock is a subset where underlying circulatory, cellular, and metabolic abnormalities are associated with a greater risk of mortality than sepsis alone."
    },
    {
        "source": "Surviving Sepsis Campaign 2021",
        "topic": "Sepsis — Hour-1 Bundle",
        "text": "The Surviving Sepsis Campaign Hour-1 Bundle mandates: (1) Measure lactate level; re-measure lactate if initial lactate >2 mmol/L. (2) Obtain blood cultures before administering antibiotics. (3) Administer broad-spectrum antibiotics. (4) Begin rapidly administering 30 mL/kg crystalloid for hypotension or lactate ≥4 mmol/L. (5) Apply vasopressors if patient is hypotensive during or after fluid resuscitation to maintain mean arterial pressure (MAP) ≥65 mmHg."
    },
    {
        "source": "Surviving Sepsis Campaign 2021",
        "topic": "Sepsis — Lab Indicators",
        "text": "Key laboratory indicators for sepsis include: elevated white blood cell count (WBC >12,000 or <4,000 cells/μL), elevated serum lactate (>2 mmol/L indicates hypoperfusion), elevated procalcitonin, elevated C-reactive protein, thrombocytopenia (platelets <100,000/μL), elevated creatinine (AKI as organ dysfunction marker), elevated bilirubin, and coagulation abnormalities (elevated INR/PTT)."
    },
    # ── ACUTE KIDNEY INJURY ─────────────────────────────────────────────────
    {
        "source": "KDIGO AKI Guidelines 2012 (updated 2024)",
        "topic": "AKI — Definition and Staging",
        "text": "Acute Kidney Injury (AKI) is defined as any of: increase in serum creatinine by ≥0.3 mg/dL within 48 hours, increase in serum creatinine to ≥1.5 times baseline within 7 days, or urine volume <0.5 mL/kg/h for ≥6 hours. KDIGO Stages: Stage 1 — creatinine 1.5-1.9× baseline or ≥0.3 mg/dL rise; Stage 2 — creatinine 2.0-2.9× baseline; Stage 3 — creatinine ≥3× baseline, or ≥4.0 mg/dL, or initiation of RRT."
    },
    {
        "source": "KDIGO AKI Guidelines 2012 (updated 2024)",
        "topic": "AKI — Management",
        "text": "AKI management: Discontinue nephrotoxic agents. Ensure adequate volume status and perfusion pressure. Monitor serum creatinine and urine output. Avoid hyperglycemia. Consider renal replacement therapy (RRT) for volume overload refractory to diuretics, severe hyperkalemia (K⁺ >6.5 mEq/L), metabolic acidosis (pH <7.1), or uremic symptoms. Target MAP ≥65 mmHg in vasopressor-dependent AKI."
    },
    # ── GLUCOSE / HYPERGLYCEMIA ─────────────────────────────────────────────
    {
        "source": "ADA Standards of Care 2024 — Inpatient Hyperglycemia",
        "topic": "ICU Glucose Management",
        "text": "For critically ill patients, insulin therapy should be initiated for persistent hyperglycemia starting at a threshold of >180 mg/dL. Once insulin therapy is started, a glucose range of 140–180 mg/dL is recommended for the majority of critically ill patients. More stringent goals (110–140 mg/dL) may be appropriate for select patients if achievable without significant hypoglycemia."
    },
    {
        "source": "ADA Standards of Care 2024 — Inpatient Hyperglycemia",
        "topic": "Hypoglycemia Risk in Inpatients",
        "text": "Hypoglycemia (blood glucose <70 mg/dL) is associated with adverse outcomes including mortality in critically ill patients. Severe hypoglycemia (<40 mg/dL) should be treated immediately with IV dextrose. Protocols should include automatic suspension of insulin infusions when glucose falls below 70 mg/dL, followed by repeat glucose check in 15 minutes."
    },
    # ── CARDIAC ─────────────────────────────────────────────────────────────
    {
        "source": "ACC/AHA Heart Failure Guidelines 2022",
        "topic": "Heart Failure — BNP/NT-proBNP",
        "text": "B-type natriuretic peptide (BNP) and N-terminal pro-BNP (NT-proBNP) are recommended for diagnosis of acute heart failure. BNP <100 pg/mL makes acute HF unlikely; BNP >500 pg/mL strongly supports the diagnosis. NT-proBNP cutoffs: <300 pg/mL (rule out), >900 pg/mL for age 50-75, >1800 pg/mL for age >75 (rule in). Elevated BNP is also associated with worse outcomes in sepsis and renal failure."
    },
    # ── ELECTROLYTES ─────────────────────────────────────────────────────────
    {
        "source": "UpToDate — Hyponatremia Management",
        "topic": "Hyponatremia",
        "text": "Hyponatremia is defined as serum sodium <135 mEq/L. Severe symptomatic hyponatremia (Na <125 mEq/L with symptoms) requires immediate treatment with hypertonic saline (3% NaCl). Correction rate should not exceed 10-12 mEq/L in 24 hours and 18 mEq/L in 48 hours to prevent osmotic demyelination syndrome (ODS). SIADH is the most common cause in hospitalized patients."
    },
    {
        "source": "UpToDate — Hyperkalemia Management",
        "topic": "Hyperkalemia",
        "text": "Hyperkalemia is defined as serum potassium >5.5 mEq/L. Severe hyperkalemia (K⁺ >6.5 mEq/L or ECG changes) requires immediate treatment: calcium gluconate IV for cardiac membrane stabilization, insulin + dextrose to shift K⁺ intracellularly, sodium bicarbonate if acidotic, albuterol nebulization, and kayexalate or patiromer for elimination. Dialysis for refractory cases."
    },
    # ── RESPIRATORY ──────────────────────────────────────────────────────────
    {
        "source": "ARDS Network / ARDSNET Protocol",
        "topic": "ARDS — Berlin Definition and Ventilation",
        "text": "ARDS (Acute Respiratory Distress Syndrome) Berlin Definition: onset within 1 week of known clinical insult, bilateral opacities on CXR/CT, respiratory failure not fully explained by cardiac failure, PaO2/FiO2 ratio: Mild 201-300, Moderate 101-200, Severe ≤100 mmHg with PEEP ≥5 cmH2O. Low tidal volume ventilation (6 mL/kg IBW) reduces mortality. Target plateau pressure <30 cmH2O. Prone positioning recommended for severe ARDS (PaO2/FiO2 <150)."
    },
]

print(f"✅ Medical guidelines corpus: {len(MEDICAL_GUIDELINES)} chunks loaded")
print(f"\nTopics covered:")
for g in MEDICAL_GUIDELINES:
    print(f"  [{g['source'].split(' — ')[0][:30]}] {g['topic']}")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="")

def get_openai_embeddings(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([r.embedding for r in response.data], dtype=np.float32)
texts = [g["text"] for g in MEDICAL_GUIDELINES] 
embeddings = get_openai_embeddings(texts)
dim = embeddings.shape[1]  # 1536 for text-embedding-3-small
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

faiss.write_index(index, str(FAISS_PATH)) 
with open(META_PATH, "wb") as f:
    pickle.dump(MEDICAL_GUIDELINES, f)

print(f"✅ FAISS index built with OpenAI embeddings: {index.ntotal} vectors, {dim} dims")

In [ ]:
def vector_search(query: str, k: int = 3) -> list[dict]:
    query_vec = get_openai_embeddings([query])
    distances, indices = index.search(query_vec, k)
    
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        chunk = MEDICAL_GUIDELINES[idx].copy()
        chunk["rank"] = rank + 1
        chunk["distance"] = float(dist)
        results.append(chunk)
    return results

In [ ]:
print("="*65)
print("🏥 MIRA DATA LAYER — END-TO-END SANITY CHECK")
print("="*65)

# Step 1: Get a real patient with abnormal labs from SQLite
print("\n📊 STEP 1: SQL Query — Fetch patient with abnormal labs")
patient_df = pd.read_sql_query("""
    SELECT p.subject_id, p.gender, p.anchor_age,
           d.label AS lab_name, l.value, l.valueuom, l.flag, l.charttime
    FROM labevents l
    JOIN patients p ON l.subject_id = p.subject_id
    JOIN d_labitems d ON l.itemid = d.itemid
    WHERE l.flag = 'abnormal'
    LIMIT 5
""", conn)

print(patient_df.to_string(index=False))

# Step 2: For each abnormal lab, find the relevant guideline
print("\n📚 STEP 2: Vector Search — Find guidelines for these abnormal labs")
if len(patient_df) > 0:
    # Use the first abnormal lab as our search query
    first_lab = patient_df.iloc[0]
    query = f"abnormal {first_lab['lab_name']} value {first_lab['value']} {first_lab.get('valueuom', '')}"
    print(f"\nSearching guidelines for: '{query}'")
    
    results = vector_search(query, k=2)
    for r in results:
        print(f"\n✅ Found: [{r['source']}] — {r['topic']}")
        print(f"   {r['text'][:300]}...")

print("\n" + "="*65)
print("✅ Data Layer is fully operational!")
print("   SQLite DB   → ready for Agent 1 (SQL Data Extractor)")
print("   FAISS Index → ready for Agent 2 (Semantic Cross-Ref)")
print("   Next: 02_tools_and_mcp.ipynb — wrap these as LangGraph Tools")
print("="*65)